In [3]:
from sage.all import *
from itertools import combinations

def incidence_to_primal_matrices(incidence_matrix):
    num_hyperedges = incidence_matrix.nrows()
    num_vertices = incidence_matrix.ncols()
    
    # List to store edges of the primal graph
    primal_edges = []
    
    for i in range(num_hyperedges):
        vertices_in_hyperedge = [j for j in range(num_vertices) if incidence_matrix[i, j] == 1]
        for u in range(len(vertices_in_hyperedge)):
            for v in range(u + 1, len(vertices_in_hyperedge)):
                edge = sorted([vertices_in_hyperedge[u], vertices_in_hyperedge[v]])
                if edge not in primal_edges:
                    primal_edges.append(edge)
    
    # Create the incidence matrix for the primal graph
    primal_incidence_matrix = Matrix(ZZ, len(primal_edges), num_vertices, 0)
    for e, (u, v) in enumerate(primal_edges):
        primal_incidence_matrix[e, u] = 1
        primal_incidence_matrix[e, v] = 1
    
    # Create the adjacency matrix for the primal graph
    primal_adjacency_matrix = Matrix(ZZ, num_vertices, num_vertices, 0)
    for u, v in primal_edges:
        primal_adjacency_matrix[u, v] = 1
        primal_adjacency_matrix[v, u] = 1
    
    return primal_incidence_matrix, primal_adjacency_matrix

def count_complete_subgraphs(primal_matrix, n):
    num_edges = primal_matrix.nrows()
    num_vertices = primal_matrix.ncols()
    
    count = 0
    complete_subgraphs = []
    
    for subgraph_edges in combinations(range(num_edges), n):
        vertices_in_subgraph = set()
        for edge in subgraph_edges:
            for v in range(num_vertices):
                if primal_matrix[edge, v] == 1:
                    vertices_in_subgraph.add(v)
        if len(vertices_in_subgraph) == n and all(
            sum(primal_matrix[edge, v] for edge in subgraph_edges) >= 2 for v in vertices_in_subgraph):
            count += 1
            complete_subgraphs.append(subgraph_edges)
    
    return count, complete_subgraphs

def count_p4_paths(primal_matrix):
    num_edges = primal_matrix.nrows()
    num_vertices = primal_matrix.ncols()
    
    count = 0
    
    for subgraph_edges in combinations(range(num_edges), 3):
        vertices_in_subgraph = set()
        for edge in subgraph_edges:
            for v in range(num_vertices):
                if primal_matrix[edge, v] == 1:
                    vertices_in_subgraph.add(v)
        if len(vertices_in_subgraph) == 4 and sum(primal_matrix[edge, v] for edge in subgraph_edges for v in vertices_in_subgraph) == 6:
            count += 1
    
    return count

def identify_simplicial_vertices(primal_matrix):
    num_edges = primal_matrix.nrows()
    num_vertices = primal_matrix.ncols()
    
    simplicial_vertices = []
    for v in range(num_vertices):
        neighbors = set(edge for edge in range(num_edges) if primal_matrix[edge, v] == 1)
        if all(
            any(primal_matrix[edge, v_prime] == 1 for edge in neighbors) for v_prime in range(num_vertices) if v_prime != v):
            simplicial_vertices.append(v)
    
    return simplicial_vertices

def find_all_cycles(incidence_matrix):
    edges = []
    num_vertices = incidence_matrix.ncols()
    
    # Create edge list from incidence matrix
    for j in range(num_vertices):
        vertices_in_edge = [i for i in range(incidence_matrix.nrows()) if incidence_matrix[i, j] == 1]
        edges.append(vertices_in_edge)
    
    # Create graph from edges
    G = Graph()
    for edge in edges:
        for i in range(len(edge)):
            for j in range(i + 1, len(edge)):
                G.add_edge(edge[i], edge[j])
    
    # Find all cycles in the graph
    cycles = G.cycle_basis()
    
    return cycles







# Define the incidence matrix
incidence_matrix = Matrix([
    [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1]])

# Get the primal graph incidence and adjacency matrices
primal_incidence_matrix, primal_adjacency_matrix = incidence_to_primal_matrices(incidence_matrix)
print("Primal graph incidence matrix:")
print(primal_incidence_matrix)

print("Primal graph adjacency matrix:")
print(primal_adjacency_matrix)

# Check for all K_n (complete subgraphs) in the primal graph
for n in range(2, primal_incidence_matrix.ncols() + 1):
    num_K_n, complete_subgraphs = count_complete_subgraphs(primal_incidence_matrix, n)
    print(f"Number of K_{n} in the primal graph:", num_K_n)
    
    # Verify if all K_n subgraphs correspond to a hyperedge
    violations = complete_subgraphs
    if violations:
        print(f"The following K_{n} subgraphs do not correspond to a hyperedge in the primal graph:")
        print(violations)
    else:
        print(f"All K_{n} subgraphs correspond to a hyperedge in the primal graph.")

num_p4 = count_p4_paths(primal_incidence_matrix)
print("Number of P_4 paths in the primal graph:", num_p4)



Primal graph incidence matrix:
[1 1 0 0 0 0 0 0 0 0 0 0]
[0 1 1 0 0 0 0 0 0 0 0 0]
[0 0 1 1 0 0 0 0 0 0 0 0]
[1 0 0 1 0 0 0 0 0 0 0 0]
[0 0 0 0 1 1 0 0 0 0 0 0]
[0 0 0 0 0 1 1 0 0 0 0 0]
[0 0 0 0 0 0 1 1 0 0 0 0]
[0 0 0 0 1 0 0 1 0 0 0 0]
[0 0 0 0 0 0 0 0 1 1 0 0]
[0 0 0 0 0 0 0 0 0 1 1 0]
[0 0 0 0 0 0 0 0 0 0 1 1]
[0 0 0 0 0 0 0 0 1 0 0 1]
Primal graph adjacency matrix:
[0 1 0 1 0 0 0 0 0 0 0 0]
[1 0 1 0 0 0 0 0 0 0 0 0]
[0 1 0 1 0 0 0 0 0 0 0 0]
[1 0 1 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 1 0 1 0 0 0 0]
[0 0 0 0 1 0 1 0 0 0 0 0]
[0 0 0 0 0 1 0 1 0 0 0 0]
[0 0 0 0 1 0 1 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 1 0 1]
[0 0 0 0 0 0 0 0 1 0 1 0]
[0 0 0 0 0 0 0 0 0 1 0 1]
[0 0 0 0 0 0 0 0 1 0 1 0]
Number of K_2 in the primal graph: 0
All K_2 subgraphs correspond to a hyperedge in the primal graph.
Number of K_3 in the primal graph: 0
All K_3 subgraphs correspond to a hyperedge in the primal graph.
Number of K_4 in the primal graph: 3
The following K_4 subgraphs do not correspond to a hyperedge in the prima

In [2]:
def unimod_check(M):
    from sage.all import Matrix, ZZ
    nrows, ncols = M.nrows(), M.ncols()
    total_unimod = True
    unimod = True

    def get_square_submatrices(matrix, size):
        submatrices = []
        for i in range(nrows - size + 1):
            for j in range(ncols - size + 1):
                submatrix = matrix.submatrix(i, j, size, size)
                submatrices.append(submatrix)
        return submatrices

    for size in range(1, min(nrows, ncols) + 1):
        for submatrix in get_square_submatrices(M, size):
            det = submatrix.det()
            if det not in [-1, 0, 1]:
                total_unimod = False
                unimod = False
                return 'not unimodular', submatrix
            elif det != 0:
                unimod = True

    if total_unimod:
        return 'totally unimodular', None
    elif unimod:
        return 'unimodular', None
    else:
        return 'not unimodular', None

def check_conformal(M):
    from sage.graphs.graph import Graph
    nrows, ncols = M.nrows(), M.ncols()
    primal_G = Graph()
    primal_G.add_vertices(range(ncols))
    
    for i in range(nrows):
        hyperedge = [j for j in range(ncols) if M[i, j] == 1]
        if len(hyperedge) > 1:
            primal_G.add_clique(hyperedge)
    
    for clique in primal_G.cliques_maximal():
        clique_set = set(clique)
        if not any(clique_set.issubset(set([j for j in range(ncols) if M[i, j] == 1])) for i in range(nrows)):
            return False, clique_set  # Return the failing maximal clique
    return True, None

def check_balanced(M):
    from sage.graphs.graph import Graph
    nrows, ncols = M.nrows(), M.ncols()
    primal_G = Graph()
    primal_G.add_vertices(range(ncols))
    
    for i in range(nrows):
        hyperedge = [j for j in range(ncols) if M[i, j] == 1]
        if len(hyperedge) > 1:
            primal_G.add_clique(hyperedge)
    
    cycle_basis = primal_G.cycle_basis()

    # Check for odd-length cycles of length >= 4
    for cycle in cycle_basis:
        if len(cycle) % 2 == 1 and len(cycle) >= 4:
            return False
    return True

def find_induced_p4(G):
    # Construct a path graph P4
    P4 = Graph([(0, 1), (1, 2), (2, 3)])  # P4 has edges (0-1), (1-2), (2-3)
    
    # Search for an induced subgraph isomorphic to P4
    for path in G.subgraph_search_iterator(P4, induced=True):
        return path
    return None

def is_cograph(M):
    from sage.graphs.graph import Graph
    nrows, ncols = M.nrows(), M.ncols()
    primal_G = Graph()
    primal_G.add_vertices(range(ncols))
    
    for i in range(nrows):
        hyperedge = [j for j in range(ncols) if M[i, j] == 1]
        if len(hyperedge) > 1:
            primal_G.add_clique(hyperedge)
    
    # Check if the primal graph is a cograph
    # Cographs are characterized by having no induced P4.
    p4 = find_induced_p4(primal_G)
    if p4 is None:
        return True, "The graph is a cograph. It does not contain any induced P4."
    else:
        return False, f"Induced P4 found: {p4}"

def hypergraph_analysis(M):
    unimod_status, submatrix_fail = unimod_check(M)
    conformal_status, failing_clique = check_conformal(M)
    balanced_status = check_balanced(M)
    cograph_status, cograph_structure = is_cograph(M)
    
    # Prepare the output regarding cograph status
    cograph_structure_str = cograph_structure
    
    return {
        'unimodularity_status': unimod_status,
        'failing_submatrix': submatrix_fail if unimod_status == 'not unimodular' else None,
        'is_conformal': conformal_status,
        'failing_clique': failing_clique if not conformal_status else None,
        'is_balanced': balanced_status,
        'is_cograph': cograph_status,
        'cograph_structure': cograph_structure_str
    }

# Example usage:
inc_matrix = Matrix(ZZ, [[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1]])

results = hypergraph_analysis(inc_matrix)
print(results)


{'unimodularity_status': 'totally unimodular', 'failing_submatrix': None, 'is_conformal': True, 'failing_clique': None, 'is_balanced': True, 'is_cograph': True, 'cograph_structure': 'The graph is a cograph. It does not contain any induced P4.'}


In [1]:


# AUTOMORPHISM GENERATORS FOR HYPERGRAPH (Manually erase ZERO COLUMNS)
# Run on: https://sagecell.sagemath.org/

from sage.combinat.designs.incidence_structures import IncidenceStructure
from sage.groups.perm_gps.permgroup_named import SymmetricGroup
import time

def binary_matrix_to_blocks(matrix):
    print("Converting binary matrix to blocks...")
    blocks = []
    for row in matrix:
        block = [i for i, x in enumerate(row) if x == 1]
        blocks.append(block)
    return blocks

def hyp_automorphism(incidence_matrix):
    print("Computing the automorphism group...")
    num_vertices = len(incidence_matrix[0])
    print("Number of vertices:", num_vertices)
    points = list(range(num_vertices))
    blocks = binary_matrix_to_blocks(incidence_matrix)
    # Add singleton blocks for vertices not in any block
    for j in range(num_vertices):
        if all(row[j] == 0 for row in incidence_matrix):
            blocks.append([j])
    print("Blocks computed:", blocks)
    incidence_structure = IncidenceStructure(points, blocks)
    print("Incidence structure created:")
    aut_g = incidence_structure.automorphism_group()
    print("Automorphism group calculated.")
    return aut_g

def S_n_generators(aut_g):
    print("Generating symmetric group generators...")
    num_vertices = aut_g.degree()
    sym_group = SymmetricGroup(num_vertices)
    generators = []
    for perm in aut_g.gens():
        # Convert permutation to cycle notation with numbers 1-8
        perm_string = perm.cycle_string()
        # Replace numbers in the cycle notation to start from 1
        perm_string = ''.join(str(int(ch) + 1) if ch.isdigit() else ch for ch in perm_string)
        generators.append(perm_string)
    return generators

def vertex_orders(incidence_matrix):
    print("Calculating vertex orders...")
    vertex_order_dict = {}
    for j in range(len(incidence_matrix[0])):
        order = sum(row[j] for row in incidence_matrix)
        vertex_order_dict[j + 1] = order  # Use numbers starting from 1
    return vertex_order_dict

def edge_orders(incidence_matrix):
    print("Calculating edge orders...")
    edge_order_dict = {}
    for i, row in enumerate(incidence_matrix):
        order = sum(row)
        edge_order_dict[i + 1] = order  # Use numbers starting from 1
    return edge_order_dict

print("Starting script...")

# Define your incidence matrix here
incidence_matrix = [[1,1,1,0,0,0,0,0,0,0,0,0,0,0],
 [1,0,0,1,1,0,0,0,0,0,0,0,0,0],
 [1,0,0,0,0,1,1,0,0,0,0,0,0,0],
 [0,1,0,1,0,1,0,0,0,0,0,0,0,0],
 [0,1,0,0,1,0,1,0,0,0,0,0,0,0],
 [0,0,1,1,0,0,1,0,0,0,0,0,0,0],
 [0,0,1,0,1,1,0,0,0,0,0,0,0,0],
 [0,0,0,0,0,0,0,1,1,1,0,0,0,0],
 [0,0,0,0,0,0,0,1,0,0,1,1,0,0],
 [0,0,0,0,0,0,0,1,0,0,0,0,1,1],
 [0,0,0,0,0,0,0,0,1,0,1,0,1,0],
 [0,0,0,0,0,0,0,0,0,1,0,1,0,1],
 [0,0,0,0,0,0,0,0,1,1,0,0,0,1],
 [0,0,0,0,0,0,0,0,0,0,1,1,1,0]]










print("Defined incidence matrix.")

start_time = time.time()
automorphisms = hyp_automorphism(incidence_matrix)  # Ensure this call assigns a value to 'automorphisms'
end_time = time.time()

print(f"Automorphism group computation took {end_time - start_time:.2f} seconds.")

if automorphisms is not None:
    # Check if the automorphism group is trivial
    if automorphisms.order() == 1:
        print("The automorphism group is trivial.")
    else:
        print("Automorphism group is non-trivial.")
        print("Automorphism group generators are:")
        symmetric_generators = S_n_generators(automorphisms)
        for gen in symmetric_generators:
            print(gen)

        # Print the cardinality of the automorphism group
        print("Cardinality of aut:", automorphisms.cardinality())

        # Print the orders of the vertices
        vertex_order_dict = vertex_orders(incidence_matrix)
        print("Vertex orders:")
        for vertex, order in vertex_order_dict.items():
            print(f"Vertex {vertex} has degree {order}")

        # Print the orders of the edges
        edge_order_dict = edge_orders(incidence_matrix)
        print("Edge orders:")
        for edge, order in edge_order_dict.items():
            vertices = [i + 1 for i, x in enumerate(incidence_matrix[edge - 1]) if x == 1]
            print(f"Edge e{edge} degree {order} {vertices}")

        # Print the degrees of the vertices with their corresponding edges
        vertex_edge_dict = {i + 1: [] for i in range(len(incidence_matrix[0]))}
        for i, row in enumerate(incidence_matrix):
            for j, val in enumerate(row):
                if val == 1:
                    vertex_edge_dict[j + 1].append(i + 1)
        print("Vertices and their edges:")
        for vertex, edges in vertex_edge_dict.items():
            print(f"Vertex {vertex} has degree {len(edges)} {['e' + str(e) for e in edges]}")

        # GAP Part: Group Structure and Normal Subgroups
        from sage.interfaces.gap import gap

        gap.eval('LoadPackage("sonata");')
        gap.eval('LoadPackage("grape");')

        def group_structure_description(generators, expected_cardinality):
            gap_generators = gap(generators)
            gap_group = gap.Group(gap_generators)
            actual_cardinality = gap.Size(gap_group)

            if actual_cardinality != expected_cardinality:
                print(f"Warning: The actual cardinality {actual_cardinality} does not match the expected cardinality {expected_cardinality}")
                subgroups = gap.Subgroups(gap_group)
                return [gap.Size(sg) for sg in subgroups if gap.Size(sg) == expected_cardinality]
            else:
                print(f"Cardinality matches expected value of {expected_cardinality}")

            structure_description = gap.StructureDescription(gap_group)
            normal_subgroups = gap.NormalSubgroups(gap_group)
            
            return structure_description, normal_subgroups

        group_info = group_structure_description(symmetric_generators, automorphisms.cardinality())
        
        if isinstance(group_info, list):
            print("Subgroups with matching cardinality:", group_info)
        else:
            structure_description, normal_subgroups = group_info
            print(f"Group Structure: {structure_description}")
            print("Normal Subgroups:")
            for nsg in normal_subgroups:
                print(f"  {gap.StructureDescription(nsg)}, Cardinality: {gap.Size(nsg)}")
            

else:
    print("Automorphism group could not be computed.")

print("Script finished.")


Starting script...
Defined incidence matrix.
Computing the automorphism group...
Number of vertices: 14
Converting binary matrix to blocks...
Blocks computed: [[0, 1, 2], [0, 3, 4], [0, 5, 6], [1, 3, 5], [1, 4, 6], [2, 3, 6], [2, 4, 5], [7, 8, 9], [7, 10, 11], [7, 12, 13], [8, 10, 12], [9, 11, 13], [8, 9, 13], [10, 11, 12]]
Incidence structure created:
Automorphism group calculated.
Automorphism group computation took 0.01 seconds.
Automorphism group is non-trivial.
Automorphism group generators are:
Generating symmetric group generators...
(9,22)(10,21)(23,24)
(4,5)(6,7)
(4,7)(5,6)
(2,3)(6,7)
(2,4)(3,5)
(1,2)(4,7)
Cardinality of aut: 336
Calculating vertex orders...
Vertex orders:
Vertex 1 has degree 3
Vertex 2 has degree 3
Vertex 3 has degree 3
Vertex 4 has degree 3
Vertex 5 has degree 3
Vertex 6 has degree 3
Vertex 7 has degree 3
Vertex 8 has degree 3
Vertex 9 has degree 3
Vertex 10 has degree 3
Vertex 11 has degree 3
Vertex 12 has degree 3
Vertex 13 has degree 3
Vertex 14 has degre

In [ ]:


#Aut Graph

def parse_cycle(cycle_str):
    from sage.groups.perm_gps.permgroup_named import SymmetricGroup
    cycle_str = cycle_str.replace('(', '').replace(')', ' ').strip()
    cycles = [tuple(map(int, cycle.split(','))) for cycle in cycle_str.split()]
    n = max(sum(cycles, ()))  # Determine the largest vertex index to define the symmetric group
    sym_group = SymmetricGroup(n)
    return sym_group(cycles)

def aut_adj_matrix(inc_matrix, gens, card):
    from sage.groups.perm_gps.permgroup import PermutationGroup
    from sage.matrix.constructor import Matrix, ZZ
    
    perms = [parse_cycle(g) for g in gens]
    G = PermutationGroup(perms)
    
    calculated_cardinality = G.order()
    if calculated_cardinality != card:
        raise ValueError("Cardinality mismatch.")
    
    group_structure = G.structure_description()
    
    n = inc_matrix.ncols()
    adj = Matrix(ZZ, n, n, 0)
    
    for perm in G:
        for i in range(n):
            for j in range(i + 1, n):
                if perm(i + 1) == j + 1:
                    adj[i, j] = 1
                    adj[j, i] = 1
    
    return adj, group_structure

inc_matrix = Matrix([[1,1,1,0,0,0,0,0,0,0,0,0,0,0],
 [1,0,0,1,1,0,0,0,0,0,0,0,0,0],
 [1,0,0,0,0,1,1,0,0,0,0,0,0,0],
 [0,1,0,1,0,1,0,0,0,0,0,0,0,0],
 [0,1,0,0,1,0,1,0,0,0,0,0,0,0],
 [0,0,1,1,0,0,1,0,0,0,0,0,0,0],
 [0,0,1,0,1,1,0,0,0,0,0,0,0,0],
 [0,0,0,0,0,0,0,1,1,1,0,0,0,0],
 [0,0,0,0,0,0,0,1,0,0,1,1,0,0],
 [0,0,0,0,0,0,0,1,0,0,0,0,1,1],
 [0,0,0,0,0,0,0,0,1,0,1,0,1,0],
 [0,0,0,0,0,0,0,0,0,1,0,1,0,1],
 [0,0,0,0,0,0,0,0,1,1,0,0,0,1],
 [0,0,0,0,0,0,0,0,0,0,1,1,1,0]]

)

gens = ['(3,5)(6,7)','(3,6)(5,7)','(2,3,6)(4,7,5)','(2,4)(5,6)','(1,2)(3,6)']
card = Integer(168)

adj_matrix, group_structure = aut_adj_matrix(inc_matrix, gens, card)

adj_matrix, group_structure


In [113]:

#Dedekind n=4

from sage.groups.perm_gps.permgroup import PermutationGroup
from sage.groups.perm_gps.permgroup_named import SymmetricGroup
from sage.matrix.constructor import Matrix

# Function to generate the incidence matrix of a Sperner hypergraph
def generate_sperner_hypergraph(generators, num_rows):
    num_vertices = 8  # Assuming 8 variables
    perms = [PermutationGroupElement(g) for g in generators]
    G = PermutationGroup(perms)
    
    # Initialize the incidence matrix
    incidence_matrix = [[0] * num_vertices for _ in range(num_rows)]
    
    for i in range(num_rows):
        perm = G.random_element()
        edge = [perm(j) % num_vertices for j in range(num_vertices)]
        edge = list(set(edge))
        if len(edge) > 1:  # Ensure that edges have more than one element to be a hyperedge
            for vertex in edge:
                incidence_matrix[i][vertex] = 1
    
    # Ensure the hypergraph is Sperner
    incidence_matrix = ensure_sperner(incidence_matrix)
    
    return incidence_matrix

# Function to ensure the hypergraph is Sperner
def ensure_sperner(matrix):
    rows_to_remove = set()
    for i in range(len(matrix)):
        for j in range(len(matrix)):
            if i != j and all(x <= y for x, y in zip(matrix[i], matrix[j])):
                rows_to_remove.add(i)
                break
    
    return [row for idx, row in enumerate(matrix) if idx not in rows_to_remove]

# Example generators (Replace with your generators)
generators = ['(1,2)(3,4)', '(5,6)', '(1,3,5,7)(2,4,6,8)']

# Number of rows in the incidence matrix
num_rows = 10

# Generate the incidence matrix
incidence_matrix = generate_sperner_hypergraph(generators, num_rows)

# Print the incidence matrix
for row in incidence_matrix:
    print(row)


[1, 1, 1, 1, 1, 1, 1, 1]


Primal graph adjacency matrix of Input:
[0 1 1 1 1 1 1]
[1 0 1 1 1 1 1]
[1 1 0 1 1 1 1]
[1 1 1 0 1 1 1]
[1 1 1 1 0 1 1]
[1 1 1 1 1 0 1]
[1 1 1 1 1 1 0]
Number of K_n in the primal graph: 35
Number of P_4 paths in the primal graph: 0
Number of K_n,n in the primal graph: 0
Disjoint cycles in the incidence graph:
[[0, 5, 6]]
Any cycles in the incidence graph:
[[0, 5, 6], [1, 5, 6], [2, 5, 6], [3, 5, 6], [4, 5, 6], [0, 4, 6], [1, 4, 6], [2, 4, 6], [3, 4, 6], [0, 3, 6], [1, 3, 6], [2, 3, 6], [0, 2, 6], [1, 2, 6], [0, 1, 6]]


In [51]:
from sage.all import *
from itertools import chain, combinations
import time

def powerset(iterable):
    """Generate all subsets of the iterable."""
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def generate_sperner_hypergraphs(n):
    """Generate all unique Sperner hypergraph structures up to isomorphism."""
    # Generate all subsets of {0, 1, ..., n-1}
    all_subsets = list(powerset(range(n)))
    # Remove the empty set
    all_subsets.remove(())
    
    # Filter to get all Sperner families (antichains)
    sperner_families = []
    for subset in powerset(all_subsets):
        subset = list(subset)
        if all(not set(s1).issubset(s2) for s1, s2 in combinations(subset, 2)):
            sperner_families.append(subset)
    
    # Convert Sperner families to incidence matrices
    incidence_matrices = []
    for family in sperner_families:
        matrix = [[0] * n for _ in range(len(family))]
        for i, edge in enumerate(family):
            for v in edge:
                matrix[i][v] = 1
        incidence_matrices.append(matrix)
    
    # Remove isomorphic duplicates
    unique_incidence_matrices = []
    for mat in incidence_matrices:
        # Create a bipartite graph for isomorphism checking
        edges = []
        for row in range(len(mat)):
            for col in range(n):
                if mat[row][col] == 1:
                    edges.append((f"r{row}", f"c{col}"))
        bipartite_graph = Graph(edges)
        # Check for isomorphisms
        if not any(bipartite_graph.is_isomorphic(Graph([(f"r{row}", f"c{col}") for row in range(len(other_mat)) for col in range(n) if other_mat[row][col] == 1])) for other_mat in unique_incidence_matrices):
            unique_incidence_matrices.append(mat)
    
    return incidence_matrices

def binary_matrix_to_blocks(matrix):
    blocks = []
    for row in matrix:
        block = [i for i, x in enumerate(row) if x == 1]
        blocks.append(block)
    return blocks

def hyp_automorphism(incidence_matrix):
    if not incidence_matrix or not incidence_matrix[0]:
        print("Empty incidence matrix encountered")
        return None

    num_vertices = len(incidence_matrix[0])
    points = list(range(num_vertices))
    blocks = binary_matrix_to_blocks(incidence_matrix)
    # Add singleton blocks for vertices not in any block
    for j in range(num_vertices):
        if all(row[j] == 0 for row in incidence_matrix):
            blocks.append([j])
    incidence_structure = IncidenceStructure(points, blocks)
    aut_g = incidence_structure.automorphism_group()
    return aut_g

def S_n_generators(aut_g):
    num_vertices = aut_g.degree()
    sym_group = SymmetricGroup(num_vertices)
    generators = []
    for perm in aut_g.gens():
        # Convert permutation to cycle notation with numbers 1-8
        perm_string = perm.cycle_string()
        # Replace numbers in the cycle notation to start from 1
        perm_string = ''.join(str(int(ch) + 1) if ch.isdigit() else ch for ch in perm_string)
        generators.append(perm_string)
    return generators

def analyze_hypergraphs(incidence_matrices):
    """Analyze incidence matrices for their automorphism groups using GAP."""
    gap.eval('LoadPackage("sonata");')
    gap.eval('LoadPackage("grape");')
    
    results = []
    
    start_time = time.time()
    for mat in incidence_matrices:
        if time.time() - start_time > 300:
            print("Computation stopped after 300 seconds.")
            break
        aut_group = hyp_automorphism(mat)
        if aut_group is None:
            continue
        if aut_group.order() == 1:
            aut_info = "trivial"
            generators = "trivial"
        else:
            aut_info = aut_group.structure_description()
            generators = S_n_generators(aut_group)
        
        results.append((mat, generators, aut_group.order(), aut_info))
    
    return results

# Example usage
n = 1
incidence_matrices = generate_sperner_hypergraphs(n)
results = analyze_hypergraphs(incidence_matrices)

# Print the total number of hypergraphs
print(f"Total number of hypergraphs: {len(results)}")

# Print the results
for mat, generators, order, aut_info in results:
    print("Incidence Matrix:")
    for row in mat:
        print(row)
    print(f"Automorphism Group Generators: {generators}")
    print(f"Automorphism Group Order: {order}")
    print(f"Automorphism Group Structure: {aut_info}")
    print()

Empty incidence matrix encountered
Total number of hypergraphs: 1
Incidence Matrix:
[1]
Automorphism Group Generators: trivial
Automorphism Group Order: 1
Automorphism Group Structure: trivial

